# 🔧 실습 2: LLM 파인튜닝 3가지 방법 비교 실험

**목표**: 사전학습된 한국어 GPT-2에 같은 데이터로 3가지 파인튜닝 방법을 적용하고,
직접 하이퍼파라미터를 바꿔가며 결과 차이를 체험합니다.

---

## 3가지 파인튜닝 방법 한눈에 보기

| 방법 | 원리 | 학습 비율 |
|------|------|----------|
| **Full Fine-tuning** | 모델 전체를 다시 학습 | 100% |
| **Adapter** | 중간에 작은 모듈을 끼워넣고 그것만 학습 | ~3~5% |
| **LoRA** | 기존 레이어에 저랭크 행렬만 추가해서 학습 | ~0.5~2% |

```
Full Fine-tuning: [████████████████] 전체를 다시 학습
Adapter:          [░░░██░░░██░░░██] 중간중간 작은 모듈만 학습
LoRA:             [░░░▪░░░▪░░░▪░░] 기존 레이어에 아주 작은 행렬만 추가
```

질문: 파라미터를 적게 학습하면 성능이 떨어질까? → 직접 확인해봅시다!

In [ ]:
# ============================================================
# 환경 설정 (Colab에서는 아래 주석 해제)
# ============================================================
# !pip install transformers torch matplotlib -q

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Config, PreTrainedTokenizerFast
import matplotlib.pyplot as plt
import numpy as np
import time
import copy

plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
# plt.rcParams['font.family'] = 'NanumGothic'    # Colab
# plt.rcParams['axes.unicode_minus'] = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {device}')
if device == 'cuda': print(f'   GPU: {torch.cuda.get_device_name(0)}')


---
## Step 1: 모델 & 데이터 준비

**모델**: `skt/kogpt2-base-v2` — SKT에서 만든 한국어 GPT-2 (125M 파라미터)

**학습 목표**: 이 모델에게 **특정 스타일** 글쓰기를 가르치기

In [ ]:
# ============================================================
# KoGPT-2 모델 로드
# ============================================================

MODEL_NAME = 'skt/kogpt2-base-v2'

tokenizer = PreTrainedTokenizerFast.from_pretrained(
    MODEL_NAME,
    bos_token='</s>', eos_token='</s>',
    unk_token='<unk>', pad_token='<pad>', mask_token='<mask>'
)

base_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
base_model.eval()

total_params = sum(p.numel() for p in base_model.parameters())
print(f'✅ 모델 로드 완료: {MODEL_NAME}')
print(f'   전체 파라미터: {total_params:,}개 ({total_params/1e6:.0f}M)')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  🔧🔧🔧 여기를 바꿔보세요! — 학습 데이터 🔧🔧🔧       ║
# ║                                                        ║
# ║  사극스타일, mz스타일, 사투리스타일 등으로 바꿔서              ║
# ║  모델이 다른 스타일을 학습하는지 실험해보세요!            ║
# ╚══════════════════════════════════════════════════════════╝

train_texts = [
    "옛날 옛적에 깊은 숲속에 작은 토끼 한 마리가 살았습니다. 토끼는 매일 아침 숲속 친구들에게 인사를 했습니다.",
    "어느 날 토끼는 반짝이는 돌멩이를 발견했습니다. 그 돌멩이는 달빛 아래에서 무지개빛으로 빛났습니다.",
    "숲속의 현명한 부엉이 할아버지가 말했습니다. 그것은 소원을 들어주는 마법의 돌이란다.",
    "토끼는 첫 번째 소원으로 모든 친구들이 행복하게 해달라고 빌었습니다. 그러자 숲 전체에 꽃이 피어났습니다.",
    "다람쥐는 도토리를 모으며 노래를 불렀습니다. 겨울이 오기 전에 충분히 모아야 한다고 생각했습니다.",
    "작은 개구리는 연못에서 뛰어놀다가 예쁜 수련꽃을 발견했습니다. 꽃잎 위에 앉아 하늘을 바라보았습니다.",
    "숲속 동물들은 매년 가을에 축제를 열었습니다. 모두 함께 노래하고 춤추며 즐거운 시간을 보냈습니다.",
    "아기 사슴은 처음으로 숲 밖을 구경했습니다. 넓은 들판과 푸른 하늘에 감탄하며 뛰어다녔습니다.",
    "할머니 거북이는 느리지만 꾸준히 걸었습니다. 서두르지 않아도 결국 목적지에 도착한다는 것을 알고 있었습니다.",
    "밤이 되자 반딧불이들이 숲을 밝혔습니다. 마치 하늘의 별들이 땅으로 내려온 것 같았습니다.",
    "비가 온 뒤 무지개가 숲 위에 걸렸습니다. 동물 친구들은 모두 나와서 무지개를 바라보며 환호했습니다.",
    "겨울이 오자 눈이 소복이 쌓였습니다. 토끼와 다람쥐는 눈사람을 만들며 깔깔 웃었습니다.",
]

# 토크나이징
train_data = [tokenizer.encode(t, return_tensors='pt').squeeze(0) for t in train_texts]

print(f'학습 데이터: {len(train_texts)}개 문장')
print(f'예시: "{train_texts[0][:50]}..."')

In [ ]:
# ============================================================
# 공통 도구: 학습 함수 & 생성 함수
# ============================================================

def train_model(model, train_data, trainable_params, num_epochs=3, lr=5e-5):
    """공통 학습 함수"""
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.01)
    model.train()
    losses = []
    t0 = time.time()
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        for ids in train_data:
            ids = ids.unsqueeze(0).to(device)
            loss = model(input_ids=ids, labels=ids).loss
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        
        avg = epoch_loss / len(train_data)
        losses.append(avg)
        print(f'    Epoch {epoch+1}/{num_epochs} | Loss: {avg:.4f}')
    
    model.eval()
    return losses, time.time() - t0


def generate_text(model, prompt, max_len=80, temperature=0.8):
    """텍스트 생성"""
    model.eval()
    ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=max_len, temperature=temperature,
            do_sample=True, top_k=50, top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


# 원본 모델 생성 결과 미리 확인
print('📝 원본 모델 (파인튜닝 전):')
print(f'   → {generate_text(base_model, "옛날 옛적에")[:100]}')
print('\n   → 한국어는 가능하지만 무색무취 스타일!')

---
## Step 2: 3가지 파인튜닝 방법 실행

### 방법 1: Full Fine-tuning
사전학습된 모델의 **모든 파라미터**를 다시 학습합니다.

장점: 성능 최고 | 단점: GPU 메모리를 가장 많이 먹음

In [ ]:
# ============================================================
# 방법 1: Full Fine-tuning
# ============================================================

# ╔══════════════════════════════════════════════════════════╗
# ║  🔧🔧🔧 바꿔볼 것: num_epochs, lr 🔧🔧🔧             ║
# ║  epochs를 1 vs 3 vs 10으로 바꿔서 비교해보세요!         ║
# ╚══════════════════════════════════════════════════════════╝

FULL_FT_EPOCHS = 3    # 🔧 바꿔보세요! (1, 3, 5, 10)
FULL_FT_LR = 5e-5     # 🔧 바꿔보세요! (1e-5, 5e-5, 1e-4)

print('🔄 방법 1: Full Fine-tuning')
print('─' * 50)

model_fullft = copy.deepcopy(base_model)
fullft_params = list(model_fullft.parameters())
fullft_trainable = sum(p.numel() for p in fullft_params)
print(f'  학습 파라미터: {fullft_trainable:,}개 (100%)')

losses_fullft, time_fullft = train_model(
    model_fullft, train_data, fullft_params,
    num_epochs=FULL_FT_EPOCHS, lr=FULL_FT_LR
)
print(f'  ⏱️ {time_fullft:.1f}초')

### 방법 2: Adapter

원본 모델은 **완전히 고정(frozen)**하고, 각 Transformer 블록에 **작은 병목 모듈**을 끼워넣어 그것만 학습합니다.
![image](https://moonlight-paper-snapshot.s3.ap-northeast-2.amazonaws.com/arxiv/beyond-inter-item-relations-dynamic-adaption-for-enhancing-llm-based-sequential-recommendation-1.png)
```
Adapter 구조:
입력 (768차원) → 축소 (768→32) → ReLU → 확대 (32→768) → + 원래 입력
                  ↑ 여기만 학습!         ↑ 여기만 학습!
```

In [ ]:
# ============================================================
# Adapter 모듈 정의
# ============================================================

class AdapterModule(nn.Module):
    """Adapter: 축소 → 활성화 → 확대 → 잔차 연결"""
    def __init__(self, hidden_size, bottleneck_size=32):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck_size)
        self.up = nn.Linear(bottleneck_size, hidden_size)
        self.act = nn.ReLU()
        nn.init.zeros_(self.up.weight)  # 시작 시 원본과 동일하도록
        nn.init.zeros_(self.up.bias)
    
    def forward(self, x):
        return x + self.up(self.act(self.down(x)))


class GPT2WithAdapters(nn.Module):
    """KoGPT-2 + Adapter"""
    def __init__(self, base_model, bottleneck_size=32):
        super().__init__()
        self.model = copy.deepcopy(base_model)
        h = self.model.config.n_embd
        
        # 원본 전체 고정!
        for p in self.model.parameters():
            p.requires_grad = False
        
        # 각 블록에 Adapter 삽입
        self.adapters = nn.ModuleList([
            AdapterModule(h, bottleneck_size)
            for _ in range(self.model.config.n_layer)
        ])
        self._register_hooks()
    
    def _register_hooks(self):
        for i, block in enumerate(self.model.transformer.h):
            adapter = self.adapters[i]
            block.register_forward_hook(
                lambda module, inp, out, a=adapter: (a(out[0]),) + out[1:]
            )
    
    def forward(self, input_ids, labels=None):
        return self.model(input_ids=input_ids, labels=labels)
    
    def generate(self, *args, **kwargs):
        return self.model.generate(*args, **kwargs)

print('✅ Adapter 모듈 정의 완료')

In [ ]:
# ============================================================
# 방법 2: Adapter 학습
# ============================================================

# ╔══════════════════════════════════════════════════════════╗
# ║  🔧🔧🔧 핵심 실험! ADAPTER_SIZE를 바꿔보세요! 🔧🔧🔧  ║
# ║                                                        ║
# ║  8   → 아주 작은 Adapter (파라미터 최소)                ║
# ║  32  → 일반적인 크기                                   ║
# ║  128 → 큰 Adapter (파라미터 많음, 성능↑?)              ║
# ║                                                        ║
# ║  질문: 크기를 키우면 성능이 계속 좋아질까?               ║
# ╚══════════════════════════════════════════════════════════╝

ADAPTER_SIZE = 32     # 🔧🔧🔧 바꿔보세요! (8, 16, 32, 64, 128)
ADAPTER_EPOCHS = 3    # 🔧 바꿔보세요!
ADAPTER_LR = 1e-3     # 🔧 바꿔보세요!

print('🔄 방법 2: Adapter')
print('─' * 50)

model_adapter = GPT2WithAdapters(base_model, bottleneck_size=ADAPTER_SIZE).to(device)

adapter_params = list(model_adapter.adapters.parameters())
adapter_trainable = sum(p.numel() for p in adapter_params)
print(f'  병목 크기: {ADAPTER_SIZE}')
print(f'  학습 파라미터: {adapter_trainable:,}개 ({adapter_trainable/total_params*100:.2f}%)')

losses_adapter, time_adapter = train_model(
    model_adapter, train_data, adapter_params,
    num_epochs=ADAPTER_EPOCHS, lr=ADAPTER_LR
)
print(f'  ⏱️ {time_adapter:.1f}초')

### 방법 3: LoRA (Low-Rank Adaptation)

원본 가중치 W는 고정하고, **저랭크 행렬 A, B만 학습**합니다.

```
원래: y = W · x                    (W는 768×768 = 589,824개)
LoRA: y = W · x + (B · A) · x     (A: 768×8, B: 8×768 = 12,288개만 학습!)
              ↑ 고정     ↑ 학습
```
![image](https://substackcdn.com/image/fetch/$s_!i3QH!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F5dfbd169-eb7e-41e1-a050-556ccd6fb679_1600x672.png)

Adapter보다도 파라미터가 적고, 학습 후 원본에 합쳐서(merge) 추론 속도 저하도 없습니다.

In [ ]:
# ============================================================
# LoRA 모듈 정의
# ============================================================

class LoRALayer(nn.Module):
    """LoRA: 저랭크 행렬 A, B로 가중치 변화량을 근사"""
    def __init__(self, original_layer, rank=8, alpha=16.0):
        super().__init__()
        self.original = original_layer
        self.original.weight.requires_grad = False
        if hasattr(self.original, 'bias') and self.original.bias is not None:
            self.original.bias.requires_grad = False
        
        # KoGPT-2는 Conv1D를 사용 (nn.Linear와 shape 방향이 다름)
        if hasattr(original_layer, 'nf'):  # Conv1D
            in_f = original_layer.weight.shape[0]
            out_f = original_layer.nf
        else:  # nn.Linear
            in_f = original_layer.in_features
            out_f = original_layer.out_features
        
        self.lora_A = nn.Parameter(torch.randn(in_f, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_f))  # 0 초기화!
        self.scale = alpha / rank
    
    def forward(self, x):
        original_out = self.original(x)                        # W·x (고정)
        lora_out = (x @ self.lora_A @ self.lora_B) * self.scale  # B·A·x (학습)
        return original_out + lora_out


def apply_lora_to_model(base_model, rank=8, alpha=16.0):
    """GPT-2의 MLP 레이어에 LoRA 적용"""
    model = copy.deepcopy(base_model)
    
    for p in model.parameters():
        p.requires_grad = False
    
    lora_layers = []
    for block in model.transformer.h:
        lora_fc = LoRALayer(block.mlp.c_fc, rank=rank, alpha=alpha)
        block.mlp.c_fc = lora_fc
        lora_layers.append(lora_fc)
        
        lora_proj = LoRALayer(block.mlp.c_proj, rank=rank, alpha=alpha)
        block.mlp.c_proj = lora_proj
        lora_layers.append(lora_proj)
    
    return model, lora_layers

print('✅ LoRA 모듈 정의 완료')

In [ ]:
# ============================================================
# 방법 3: LoRA 학습
# ============================================================

# ╔══════════════════════════════════════════════════════════╗
# ║  🔧🔧🔧 핵심 실험! LORA_RANK를 바꿔보세요! 🔧🔧🔧    ║
# ║                                                        ║
# ║  2   → 극도로 작은 LoRA (파라미터 최소, 성능은?)        ║
# ║  8   → 일반적인 크기                                   ║
# ║  32  → 큰 LoRA (성능↑, 하지만 효율성은?)               ║
# ║                                                        ║
# ║  LORA_ALPHA도 바꿔보세요!                               ║
# ║  alpha = rank × 2 가 일반적인 시작점                    ║
# ║  alpha가 너무 작으면? 너무 크면?                         ║
# ╚══════════════════════════════════════════════════════════╝

LORA_RANK = 8         # 🔧🔧🔧 바꿔보세요! (2, 4, 8, 16, 32)
LORA_ALPHA = 16.0     # 🔧🔧🔧 바꿔보세요! (4, 8, 16, 32, 64)
LORA_EPOCHS = 3       # 🔧 바꿔보세요!
LORA_LR = 1e-3        # 🔧 바꿔보세요!

print('🔄 방법 3: LoRA')
print('─' * 50)

model_lora, lora_layers = apply_lora_to_model(base_model, rank=LORA_RANK, alpha=LORA_ALPHA)
model_lora = model_lora.to(device)

lora_params = [p for layer in lora_layers for p in [layer.lora_A, layer.lora_B]]
lora_trainable = sum(p.numel() for p in lora_params)
print(f'  rank: {LORA_RANK}, alpha: {LORA_ALPHA}')
print(f'  학습 파라미터: {lora_trainable:,}개 ({lora_trainable/total_params*100:.2f}%)')

losses_lora, time_lora = train_model(
    model_lora, train_data, lora_params,
    num_epochs=LORA_EPOCHS, lr=LORA_LR
)
print(f'  ⏱️ {time_lora:.1f}초')

---
## Step 3: 결과 비교!

In [ ]:
# ============================================================
# 학습 곡선 & 파라미터 비교 시각화
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Loss 곡선 ──
info = {
    'Full FT': {'losses': losses_fullft, 'color': '#e74c3c', 'params': fullft_trainable},
    f'Adapter(size={ADAPTER_SIZE})': {'losses': losses_adapter, 'color': '#2ecc71', 'params': adapter_trainable},
    f'LoRA(rank={LORA_RANK})': {'losses': losses_lora, 'color': '#3498db', 'params': lora_trainable},
}

for name, d in info.items():
    epochs = range(1, len(d['losses'])+1)
    ax1.plot(epochs, d['losses'], 'o-', label=name, color=d['color'], linewidth=2, markersize=8)

ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── 파라미터 수 비교 ──
names = list(info.keys())
params = [d['params'] for d in info.values()]
colors = [d['color'] for d in info.values()]
times = [time_fullft, time_adapter, time_lora]

bars = ax2.bar(names, [p/1e6 for p in params], color=colors, alpha=0.8)
for bar, p, t in zip(bars, params, times):
    pct = p / total_params * 100
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{p/1e6:.1f}M\n({pct:.1f}%)\n{t:.0f}s',
             ha='center', va='bottom', fontsize=10)

ax2.set_ylabel('Trainable Parameters (M)')
ax2.set_title('Parameters & Training Time')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 🧪 핵심! 같은 프롬프트로 생성 결과 비교
# ============================================================

# ╔══════════════════════════════════════════════════════════╗
# ║  🔧🔧🔧 프롬프트를 바꿔보세요! 🔧🔧🔧                 ║
# ║                                                        ║
# ║  학습 데이터에 있는 단어 vs 없는 단어로 시작하면         ║
# ║  결과가 어떻게 달라지는지 비교해보세요!                  ║
# ╚══════════════════════════════════════════════════════════╝

test_prompts = [
    "옛날 옛적에",         # 🔧🔧🔧 바꿔보세요!
    "숲속에서 토끼가",     # 🔧🔧🔧
    "어느 날 밤에",       # 🔧🔧🔧
]

all_models = {
    '원본 (파인튜닝 전)': base_model,
    'Full Fine-tuning': model_fullft,
    f'Adapter (size={ADAPTER_SIZE})': model_adapter,
    f'LoRA (rank={LORA_RANK})': model_lora,
}

for prompt in test_prompts:
    print(f'\n{"═"*70}')
    print(f'📝 프롬프트: "{prompt}"')
    print(f'{"═"*70}')
    
    for name, m in all_models.items():
        try:
            result = generate_text(m, prompt, max_len=60)
        except:
            result = '(생성 실패)'
        print(f'\n  [{name}]')
        print(f'  → {result[:120]}')

In [ ]:
# ============================================================
# 최종 비교 표
# ============================================================

print(f'\n{"═"*75}')
print(f'📊 3가지 파인튜닝 방법 최종 비교')
print(f'{"═"*75}')
print(f'{"방법":>25s} | {"학습 파라미터":>14s} | {"비율":>7s} | {"최종 Loss":>9s} | {"시간":>5s}')
print('─' * 75)

results = [
    ('Full Fine-tuning', fullft_trainable, losses_fullft[-1], time_fullft),
    (f'Adapter (size={ADAPTER_SIZE})', adapter_trainable, losses_adapter[-1], time_adapter),
    (f'LoRA (rank={LORA_RANK})', lora_trainable, losses_lora[-1], time_lora),
]

for name, p, loss, t in results:
    pct = p / total_params * 100
    print(f'{name:>25s} | {p:>11,}개 | {pct:>5.2f}% | {loss:>9.4f} | {t:>4.1f}s')

print(f'\n💡 관찰 포인트:')
print(f'  1. Full FT는 100% 학습하여 loss가 가장 낮지만 시간/메모리를 많이 씀')
print(f'  2. Adapter와 LoRA는 1~5%만 학습해도 비슷한 수준의 loss 달성')
print(f'  3. 실제 7B, 14B 모델에서는 Full FT가 불가능 → Adapter/LoRA 필수!')
print(f'\n🔧 다음 실험: 위로 돌아가서 ADAPTER_SIZE, LORA_RANK를 바꿔서 다시 실행!')

---
## 🧪 자유 실험 가이드

아래 표를 참고해서 위 코드의 값을 바꾸고 다시 실행해보세요!

### 실험 1: Adapter 크기 비교
| `ADAPTER_SIZE` | 예상 | 확인할 것 |
|------|------|------|
| 8 | 파라미터 최소, 성능은? | loss가 얼마나 높아지는지 |
| 32 | 일반적 | 기준점 |
| 128 | 파라미터 많음 | Full FT에 근접하는지 |

### 실험 2: LoRA rank 비교
| `LORA_RANK` | 예상 | 확인할 것 |
|------|------|------|
| 2 | 극소 파라미터 | 동화 스타일을 학습할 수 있는지 |
| 8 | 일반적 | 기준점 |
| 32 | 많은 파라미터 | Adapter보다 좋은지 |

### 실험 3: 데이터 스타일 변경
| `train_texts` 내용 | 확인할 것 |
|------|------|
| 동화 (기본) | "옛날 옛적에" 프롬프트에 동화 생성? |
| 뉴스 스타일로 교체 | 같은 프롬프트에서 톤이 바뀌는지 |
| 시/감성 글로 교체 | 문학적 표현이 나오는지 |

### 실험 4: 생성 프롬프트 변경
| 프롬프트 | 확인할 것 |
|------|------|
| "옛날 옛적에" | 학습 데이터에 있는 시작 → 동화 스타일? |
| "오늘 서울에서" | 학습 데이터에 없는 시작 → 여전히 동화? |
| "인공지능이란" | 완전 다른 주제 → 원본과 차이가 있는지 |

---

## ✅ 실습 2 요약

| 방법 | 학습 대상 | 장점 | 단점 |
|------|-----------|------|------|
| **Full FT** | 전체 파라미터 (100%) | 최고 성능 | GPU 메모리 최대, 원본 파괴 |
| **Adapter** | 삽입된 병목 모듈 (~3%) | 원본 보존, 모듈 교체 가능 | 추론 시 약간 느림 |
| **LoRA** | 저랭크 행렬 (~1%) | 가장 효율적, merge 가능 | rank 튜닝 필요 |

**다음 실습**: 실제 EXAONE 모델로 AI 캐릭터 챗봇 체험! →